In [2]:
import requests

PORT = 8000

END_POINT = 'chat'

url = f'http://localhost:{PORT}/{END_POINT}'

data = "Zuza meduza"

def sent_request(data):

    response = requests.post(url, json = data)

    if response == 200:
        print(response.json())
    else:
        print(response)

sent_request(data)

# response = requests.get(url)

# print(response, response.text)



<Response [422]>


In [ ]:
import requests

PORT = 8000

END_POINT = 'chat/'

url = f'http://localhost:{PORT}/{END_POINT}'

data = {'prompt': 'Bombardiro Crocodilo'}

response = requests.post(url, data=data, stream=True)  # stream=True jeśli chcesz strumieniować odpowiedź
for chunk in response.iter_content(chunk_size=1024):
    if chunk:
        print(chunk.decode(), end='')

In [13]:
import requests


PORT = 8000

END_POINT = '/add_message'
# END_POINT = '/chat'

url = f'http://localhost:{PORT}/{END_POINT}'

data = {'prompt': 'Ala ma kruka!'}

response = requests.post(url, data=data)  # stream=True jeśli chcesz strumieniować odpowiedź
# for chunk in response.iter_content(chunk_size=1024):
#     if chunk:
#         print(chunk.decode(), end='')

In [21]:
# CHECK DB Content

import sqlite3

# Path to your .sqlite file
db_path = "chat_app_messages.sqlite"

# Connect to the SQLite database
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# Get all table names (optional, for exploration)
cursor.execute("SELECT * FROM messages;")

rows = cursor.fetchall()

# Clean up
cursor.close()
conn.close()


In [20]:
rows

[(None,
  b'[{"parts":[{"content":"test","timestamp":"2025-05-21T20:02:58.644846Z","part_kind":"user-prompt"}],"instructions":null,"kind":"request"},{"parts":[{"content":"Hello! How can I assist you today?","part_kind":"text"}],"model_name":"gpt-3.5-turbo","timestamp":"2025-05-21T20:02:59Z","kind":"response"}]'),
 (None,
  b'[{"parts":[{"content":"test","timestamp":"2025-05-21T21:02:13.052559Z","part_kind":"user-prompt"}],"instructions":null,"kind":"request"},{"parts":[{"content":"All good on my end! Let me know if you have any questions or need help with anything.","part_kind":"text"}],"model_name":"gpt-3.5-turbo","timestamp":"2025-05-21T21:02:15Z","kind":"response"}]'),
 (None,
  b'[{"parts":[{"content":"You are a DevOps expert. Your task is to analyze log messages received from Kafka and provide concise, \\n                    actionable explanations or solutions for any detected issues. Focus on identifying errors, warnings, and \\n                    abnormal patterns.","timestamp

In [ ]:
from fastapi import FastAPI, Request
from pydantic import BaseModel
from pydantic_ai import Agent

app = FastAPI()

# Define the request schema
class QueryRequest(BaseModel):
    message: str

# Initialize the LLM agent
agent = Agent(model="gpt-4", system="You are a helpful assistant.")

@app.post("/chat")
async def chat(request: QueryRequest):
    response = await agent.run(request.message)
    return {"response": response}


In [14]:
# DISCORD CHANEL

webhook_id = 'FzUvToXRdk0WtCqrZhcwIgmEu-R-hoSHQc8eyFsT6OBGX8_n-zPzjNgDouDKocGlNX-w'
webhook_url = f"https://discord.com/api/webhooks/1377369991803834391/{webhook_id}"


import requests

def send_to_discord(webhook_url, message):
    requests.post(webhook_url, json={"content": message})


send_to_discord(webhook_url, "XYZ")

In [1]:

import redis

# Connect to Redis
r = redis.Redis(host='localhost', port=6379, db=0)

# Zapisz coś z TTL (np. 15 minut = 900 sekund)
r.set("log:example", "vector_string_or_serialized_data", ex=900)

# Odczytaj
value = r.get("log:example")

print("Z Redis:", value.decode() if value else "Brak klucza")


Z Redis: vector_string_or_serialized_data


In [23]:
# REDIS TESTS:

from pydantic import BaseModel
from redis import Redis
import time
import uuid
from typing import Tuple

# class LogEntry(BaseModel):
#     message: str

# Connect to Redis:
redis_db = Redis(host='localhost', port=6379, db=0)
# db=0 means default Redis logical database (database number 0).

# How long logs will stay in Redis db
LOG_TTL = 15 * 60  # 15 minutes TTL

def make_red_log_id() -> Tuple[int, str]:
    """ Function to generate log id in Redis DB"""

    # millisecond timestamp
    ms_time_stamp = int(time.time() * 1000)

    # short UUID suffix to guarantee uniqueness
    uuid_suffix = uuid.uuid4().hex[:6]

    return (ms_time_stamp, uuid_suffix)


def store_log_redis(entry):
    """store log in Redis DB"""

    # Create log
    redis_log_id = make_red_log_id()

    # timestamp in ms:
    ms_time_stamp: int = redis_log_id[0]
    suffix_id: str = redis_log_id[1]

    # Store log message with TTL
    redis_db.set(f"{ms_time_stamp}:{suffix_id}", entry, ex=LOG_TTL)

    # Add to sorted set with timestamp as score
    redis_db.zadd("logs:zset", {suffix_id: ms_time_stamp})

    return {"redis_log_id": redis_log_id, "redis_timestamp": ms_time_stamp}

log1 = '[2025-04-25 22:35:18,082] INFO Registered signal handlers for TERM, INT, HUP (org.apache.kafka.common.utils.LoggingSignalHandler)'

store_log_redis(log1)

{'redis_log_id': (1748684293087, 'e904bb'), 'redis_timestamp': 1748684293087}

In [ ]:
def get_context_logs(log_id: str):
    """Return 5 logs before given log"""

    rank = redis_db.zrank("logs:zset", log_id)
    
    if rank is None or rank == 0:
        return []

    start = max(0, rank - 5)
    prev_log_ids = redis_db.zrange("logs:zset", start, rank - 1)

    logs = []
    for lid in prev_log_ids:
        lid_str = lid.decode()
        msg = redis_db.get(f"log:{lid_str}")

        if msg:
            logs.append({"log_id": lid_str, "message": msg.decode()})

    return logs[::-1]  # newest first



get_context_logs()



[]

In [21]:
# return simple ID:

def get_context_logs(log_id: str):
    """Return given log"""

    given_log = redis_db.get(f"{log_id}")

    return given_log  # newest first

idX = (1748684058377, '3caa81')

res = get_context_logs(f'{idX[0]}:{idX[1]}').decode()

res

'[2025-04-25 22:35:18,082] INFO Registered signal handlers for TERM, INT, HUP (org.apache.kafka.common.utils.LoggingSignalHandler)'

In [24]:
keys = redis_db.keys("*")

print("All keys:")

for key in keys:
    print(key.decode())

All keys:
1748684056073:741bd2
1748684056784:8a3364
1748684168136:02833d
1748684058377:3caa81
1748684054207:e7d3ab
1748684166486:bb74c6
1748684054746:aad7db
1748684293087:e904bb
1748684057168:17f1ad
1748684052997:07f991
1748684057966:e6b7f2
1748684055680:8b8031
1748684292075:7656f6
1748684056436:8f5a2d
logs:zset
1748684055234:5a0ef2
1748684167406:2af0e3


In [16]:
keys = redis_db.keys("*")

print("All keys and values:")

for key in keys:

    key_str = key.decode()

    key_type = redis_db.type(key).decode()
    

    print(f"\nKey: {key_str} (Type: {key_type})")

    if key_type == "string":

        value = redis_db.get(key)

        print(f"  Value: {value.decode()}")

    elif key_type == "zset":

        entries = redis_db.zrange(key, 0, -1, withscores=True)

        for member, score in entries:
            print(f"  Member: {member.decode()} | Score: {int(score)}")

    else:
        print("[Unhandled or empty value]")


All keys and values:

Key: 1748684056073:741bd2 (Type: string)
  Value: [2025-04-25 22:35:18,082] INFO Registered signal handlers for TERM, INT, HUP (org.apache.kafka.common.utils.LoggingSignalHandler)

Key: 1748684057168:17f1ad (Type: string)
  Value: [2025-04-25 22:35:18,082] INFO Registered signal handlers for TERM, INT, HUP (org.apache.kafka.common.utils.LoggingSignalHandler)

Key: 1748684057966:e6b7f2 (Type: string)
  Value: [2025-04-25 22:35:18,082] INFO Registered signal handlers for TERM, INT, HUP (org.apache.kafka.common.utils.LoggingSignalHandler)

Key: 1748684052997:07f991 (Type: string)
  Value: [2025-04-25 22:35:18,082] INFO Registered signal handlers for TERM, INT, HUP (org.apache.kafka.common.utils.LoggingSignalHandler)

Key: 1748684056784:8a3364 (Type: string)
  Value: [2025-04-25 22:35:18,082] INFO Registered signal handlers for TERM, INT, HUP (org.apache.kafka.common.utils.LoggingSignalHandler)

Key: 1748684055680:8b8031 (Type: string)
  Value: [2025-04-25 22:35:18,08

In [25]:
def get_context_logs(log_id: str):
    """Return 5 logs before given log"""

    rank = redis_db.zrank("logs:zset", log_id)
    
    if rank is None or rank == 0:
        return []

    start = max(0, rank - 5)
    prev_log_ids = redis_db.zrange("logs:zset", start, rank - 1)

    logs = []
    for lid in prev_log_ids:
        lid_str = lid.decode()
        msg = redis_db.get(f"log:{lid_str}")

        if msg:
            logs.append({"log_id": lid_str, "message": msg.decode()})

    return logs[::-1]  # newest first



get_context_logs('1748684166486:bb74c6')

# 1748684166486:bb74c6
# 1748684054746:aad7db

[]

In [26]:
# import redis
# from typing import List

# def get_recent_logs_before(redis_client: redis.Redis, 
#                            zset_key: str, 
#                            ref_key: str, 
#                            count: int = 5) -> List[str]:
#     """
#     Returns up to `count` most recent logs before the given `ref_key` 
#     from a Redis sorted set.

#     Parameters:
#         redis_client: Redis client instance.
#         zset_key: The name of the sorted set (e.g., "logs").
#         ref_key: The reference log key in format "timestamp:uuid".
#         count: Number of logs to return (default: 5).

#     Returns:
#         List of log keys (strings), ordered from most recent to oldest.
#     """

#     # Extract timestamp from ref_key
#     try:
#         ref_timestamp = int(ref_key.split(":")[0])

#     except (IndexError, ValueError):
#         raise ValueError("Invalid reference key format. Expected 'timestamp:uuid'.")

#     # Get logs with score less than the reference timestamp
#     # Use ZREVRANGEBYSCORE for reverse order (most recent first)
#     logs = redis_client.zrevrangebyscore(
#         zset_key,
#         max=ref_timestamp - 1,  # strictly before ref_timestamp
#         min='-inf',
#         start=0,
#         num=count
#     )

#     return logs


1748684822.6929226

In [1]:
# REDIS TESTS:
# SIMPLIFIED


from redis import Redis
import time
from uuid import uuid4


# Connect to Redis:
redis_db = Redis(host='localhost', port=6379, db=0, decode_responses=True)
# db = 0 means default Redis logical database (database number 0).
# decode_responses=True will return decoded string

# How long logs will stay in Redis db
LOG_TTL = 15 * 60  # 15 minutes TTL

def make_red_log_id() -> tuple[int, str]:
    """ Function to generate log id in Redis DB"""

    # Microsecond timestamp
    micros_timestamp = int(time.time() * 1_000_000)

    # short UUID suffix to guarantee uniqueness
    uuid_suffix = uuid4().hex[:6]

    return micros_timestamp, uuid_suffix

def store_log_redis(entry: str) -> None:
    """Store log entry in Redis with TTL and add it to a sorted set"""

    micros_timestamp, uuid_suffix = make_red_log_id()

    redis_log_id = f"{micros_timestamp}:{uuid_suffix}"
    
    # Store the log entry with expiration
    redis_db.set(redis_log_id, entry, ex=LOG_TTL)

    # Add to sorted set for chronological lookup
    redis_db.zadd("logs:zset", {redis_log_id: micros_timestamp})

    print(f"redis_log_id :{redis_log_id}, entry: {entry}")

    return None

log1 ='[2025-04-25 22:35:18,082] INFO Registered signal handlers for TERM, INT, HUP (org.apache.kafka.common.utils.LoggingSignalHandler)'
log2 ='[2025-04-25 10:48:24,173] INFO [SocketServer listenerType=CONTROLLER, nodeId=1] Created data-plane acceptor and processors for endpoint : ListenerName(CONTROLLER_SSL) (kafka.network.SocketServer)'
log3 ='[2025-04-25 10:48:24,175] INFO [SharedServer id=1] Starting SharedServer (kafka.server.SharedServer)'
log4 ='[2025-04-25 10:48:24,242] INFO [LogLoader partition=__cluster_metadata-0, dir=/mnt/kafka-data/kafka-kraft-metadata] Recovering unflushed segment 124648021. 0/1 recovered for __cluster_metadata-0. (kafka.log.LogLoader)'
log5 ='[2025-04-25 10:48:24,245] INFO [LogLoader partition=__cluster_metadata-0, dir=/mnt/kafka-data/kafka-kraft-metadata] Loading producer state till offset 124648021 with message format version 2 (kafka.log.UnifiedLog$)'
log6 ='[2025-04-25 10:48:24,246] INFO [LogLoader partition=__cluster_metadata-0, dir=/mnt/kafka-data/kafka-kraft-metadata] Reloading from producer snapshot and rebuilding producer state from offset 124648021 (kafka.log.UnifiedLog$)'
log7 ='[2025-04-25 10:48:24,246] INFO Deleted producer state snapshot /mnt/kafka-data/kafka-kraft-metadata/__cluster_metadata-0/00000000000125835950.snapshot (org.apache.kafka.storage.internals.log.SnapshotFile)'
log8 ='[2025-04-25 10:48:24,255] INFO [ProducerStateManager partition=__cluster_metadata-0]Wrote producer snapshot at offset 124648021 with 0 producer ids in 6 ms. (org.apache.kafka.storage.internals.log.ProducerStateManager)'

logs = [log1, log2, log3, log4, log5, log6, log7, log8]

for i in logs:
    store_log_redis(i)


redis_log_id :1748687359065607:e10877, entry: [2025-04-25 22:35:18,082] INFO Registered signal handlers for TERM, INT, HUP (org.apache.kafka.common.utils.LoggingSignalHandler)
redis_log_id :1748687359071758:06fff5, entry: [2025-04-25 10:48:24,173] INFO [SocketServer listenerType=CONTROLLER, nodeId=1] Created data-plane acceptor and processors for endpoint : ListenerName(CONTROLLER_SSL) (kafka.network.SocketServer)
redis_log_id :1748687359071758:9d68ac, entry: [2025-04-25 10:48:24,175] INFO [SharedServer id=1] Starting SharedServer (kafka.server.SharedServer)
redis_log_id :1748687359071758:0b3ff6, entry: [2025-04-25 10:48:24,242] INFO [LogLoader partition=__cluster_metadata-0, dir=/mnt/kafka-data/kafka-kraft-metadata] Recovering unflushed segment 124648021. 0/1 recovered for __cluster_metadata-0. (kafka.log.LogLoader)
redis_log_id :1748687359071758:975186, entry: [2025-04-25 10:48:24,245] INFO [LogLoader partition=__cluster_metadata-0, dir=/mnt/kafka-data/kafka-kraft-metadata] Loading p

In [31]:
from typing import List

def get_logs_before(ref_log_id: str, count: int = 5) -> List[str]:
    """
    Returns up to `count` log entries from Redis that are strictly before the given `ref_log_id`.

    Parameters:
        ref_log_id (str): The reference log ID in the format "timestamp:uuid".
        count (int): Number of logs to return (default: 5).

    Returns:
        List[str]: Log entries (strings), most recent first.
    """
    
    try:
        ref_timestamp = int(ref_log_id.split(":")[0])
    except (IndexError, ValueError):
        raise ValueError("Invalid log ID format. Expected 'timestamp:uuid'.")

    # Get up to `count` entries with lower score than the reference
    log_ids = redis_db.zrevrangebyscore(
        "logs:zset",
        max=ref_timestamp - 1,
        min='-inf',
        start=0,
        num=count
    )

    # Fetch actual log values by keys
    if not log_ids:
        return []

    log_entries = redis_db.mget(log_ids)

    # Filter out expired (None) entries and zip with their keys
    return [
        entry for entry in log_entries
        if entry is not None
    ]



ref_log_id = "1748687359077255:2fd979"

# Fetch logs before this log
logs = get_logs_before(ref_log_id)

for log in logs:
    print(log)

NameError: name 'redis_db' is not defined

In [26]:
from redis import Redis


# Connect to Redis:
redis_db = Redis(host='localhost', port=6379, db=0, decode_responses=True)

redis_db.flushdb()

True

In [24]:
import redis

# Connect to Redis (adjust host/port/db as needed)
r = redis.Redis(host='localhost', port=6379, db=0)

# Get all keys
keys = r.keys()  # You can filter with a pattern like 'user:*'

for key in keys:
    print(key, r.mget(key))

b'temp_logs' [None]
b'1748793243612097:a43168' [b'{"invalid_log": "org.apache.kafka.common.errors.AuthorizerNotReadyException"}']
b'1748793248509269:7b192d' [b'{"invalid_log": "org.apache.kafka.common.errors.AuthorizerNotReadyException"}']
b'1748793341659258:5cc4b1' [b'{"invalid_log": "org.apache.kafka.common.errors.AuthorizerNotReadyException"}']
b'1748793344010282:2a1778' [b'{"valid_log": {"timestamp": "2025-04-25T10:48:40.781000", "level": "ERROR", "component": "ControllerApis nodeId=1", "message": "Unexpected error handling request RequestHeader(apiKey=FETCH, apiVersion=15, clientId=raft-client-4, correlationId=5, headerVersion=2) -- FetchRequestData(clusterId=\'9n09PkoVRJSCcZemHAmm6g\', replicaId=-1, replicaState=ReplicaState(replicaId=4, replicaEpoch=-1), maxWaitMs=500, minBytes=0, maxBytes=8388608, isolationLevel=0, sessionId=0, sessionEpoch=-1, topics=[FetchTopic(topic=\'\', topicId=AAAAAAAAAAAAAAAAAAAAAQ, partitions=[FetchPartition(partition=0, currentLeaderEpoch=4372, fetchOf

In [36]:
from redis.asyncio import Redis
import asyncio
from typing import List

# Connect to Redis:
async def redis_init() -> Redis|None:

    redis_db = Redis(host='localhost', port=6379, db=0, decode_responses=True)
    # db = 0 means default Redis logical database (database number 0).
    # decode_responses=True will return decoded string
    
    return redis_db


async def get_logs_before(redis_db: Redis, ref_log_id: str, num_of_logs: int = 5) -> List[dict]:

    """
    Returns up to 'num_of_logs' log entries from Redis that are strictly before the given 'ref_log_id'.

    Parameters:
        ref_log_id (str): The reference log ID in the format "timestamp:uuid".
        num_of_logs (int): Number of logs to return (default: 5).

    Returns:
        List[str]: Log entries (strings), most recent last.
    """
    

    try:
        ref_timestamp = int(ref_log_id.split(":")[0])
    except (IndexError, ValueError):
        raise ValueError("Invalid log ID format. Expected 'timestamp:uuid'.")

    # ascending order:
    log_ids = await redis_db.zrangebyscore(
        "temp_logs",
        min = '-inf',
        max = ref_timestamp - 1,
        start = 0,
        num = num_of_logs
    )
    
    # Fetch actual log values by keys
    if not log_ids:
        return []
    print("All log_ids:", log_ids)
    
    log_entries = await redis_db.mget(log_ids)
    print("All log_ids:", log_entries)
    
    # Return list of dicts with log_id and message, filter out missing
    recent_logs: list = [
        {"log_id": lid, "message": entry}
        for lid, entry in zip(log_ids, log_entries)
        if entry is not None
    ]
    return recent_logs
    

async def test_redis_module():

    redis_db = await redis_init()

    ref_log_id = '1748769416209411:17278e'

    return await get_logs_before(redis_db, ref_log_id, num_of_logs=5)

await test_redis_module()

All log_ids: ['1748712421240530:5dae6e', '1748712421252498:9064e5', '1748712421252498:dce7bb', '1748712421252498:f0af2d', '1748712421256496:18f863']
All log_ids: [None, None, None, None, None]


[]

In [14]:
import redis

redis_db = redis.Redis(host='localhost', port=6379, db=0, decode_responses=True)
# db = 0 means default Redis logical database (database number 0).
# decode_responses=True will return decoded string

redis_db.flushall()

True

In [ ]:
import requests


PORT = 8000

BASE_URL = f'http://localhost:{PORT}'


def delete_chat(chat_id):
    url = f"{BASE_URL}/chat/delete"
    response = requests.delete(url)

    print(f"Request to DELETE {url}")
    print(f"Status code: {response.status_code}")
    print("Response body:", response.text)


test_chat_id = "example-chat-id-123"
delete_chat(test_chat_id)


Request to DELETE http://localhost:8000/chat/example-chat-id-123
Status code: 200
Response body: "example-chat-id-123"


In [15]:
import requests

PORT = 8000

BASE_URL = f'http://localhost:{PORT}'

def delete_chats():
    url = f"{BASE_URL}/chat/delete"
    payload = {
        "msgs_ids": [102, 101]
    }

    response = requests.delete(
        url,
        json=payload,
        headers={"Content-Type": "application/json"}
    )

    print(f"Status code: {response.status_code}")
    print("Response body:", response.text)


delete_chats()


Status code: 200
Response body: {"RECEIVED: delete_ids":[102,101]}


In [ ]:
import time
import uuid

def generate_chat_id():
    timestamp = int(time.time() * 1000)
    rand_part = uuid.uuid4().hex[:7]  # 7 UUID sings
    return f"chat-{timestamp}-{rand_part}"


print(generate_chat_id())



chat-1749228882576-5b24dec
